## Loading Files

In [1]:
from safetensors import safe_open
import os

files = [
    "./my_models/model-00001-of-00002.safetensors",
    "./my_models/model-00002-of-00002.safetensors",
]

for file in files:
    print(f"\n=== {file} ===")

    with safe_open(f"{file}", framework="pt", device="cpu") as f:
        for key in f.keys():
            print(f"Name: {key}, shape:{f.get_tensor(key).shape}")


=== ./my_models/model-00001-of-00002.safetensors ===
Name: model.embed_tokens.weight, shape:torch.Size([128256, 3072])
Name: model.layers.0.input_layernorm.weight, shape:torch.Size([3072])
Name: model.layers.0.mlp.down_proj.weight, shape:torch.Size([3072, 8192])
Name: model.layers.0.mlp.gate_proj.weight, shape:torch.Size([8192, 3072])
Name: model.layers.0.mlp.up_proj.weight, shape:torch.Size([8192, 3072])
Name: model.layers.0.post_attention_layernorm.weight, shape:torch.Size([3072])
Name: model.layers.0.self_attn.k_proj.weight, shape:torch.Size([1024, 3072])
Name: model.layers.0.self_attn.o_proj.weight, shape:torch.Size([3072, 3072])
Name: model.layers.0.self_attn.q_proj.weight, shape:torch.Size([3072, 3072])
Name: model.layers.0.self_attn.v_proj.weight, shape:torch.Size([1024, 3072])
Name: model.layers.1.input_layernorm.weight, shape:torch.Size([3072])
Name: model.layers.1.mlp.down_proj.weight, shape:torch.Size([3072, 8192])
Name: model.layers.1.mlp.gate_proj.weight, shape:torch.Size

## First layer Weights

In [2]:
weights = {}

for file in files:
    with safe_open(f"./my_models/{file}", framework="pt", device="cpu") as f:
        for key in f.keys():
            if "layers.0." in key:
                weights[key] = f.get_tensor(key)

for name, tensor in weights.items():
    print(name, tensor.shape)

model.layers.0.input_layernorm.weight torch.Size([3072])
model.layers.0.mlp.down_proj.weight torch.Size([3072, 8192])
model.layers.0.mlp.gate_proj.weight torch.Size([8192, 3072])
model.layers.0.mlp.up_proj.weight torch.Size([8192, 3072])
model.layers.0.post_attention_layernorm.weight torch.Size([3072])
model.layers.0.self_attn.k_proj.weight torch.Size([1024, 3072])
model.layers.0.self_attn.o_proj.weight torch.Size([3072, 3072])
model.layers.0.self_attn.q_proj.weight torch.Size([3072, 3072])
model.layers.0.self_attn.v_proj.weight torch.Size([1024, 3072])


## Rebuilding Modules

In [2]:
import torch


class LlamaMlp(torch.nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.up_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=intermediate_size, bias=False
        )
        self.gate_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=intermediate_size, bias=False
        )
        self.down_proj = torch.nn.Linear(
            in_features=intermediate_size, out_features=hidden_size, bias=False
        )

    def forward(self, x):
        up = self.up_proj(x)
        gate = self.gate_proj(x)

        hidden = torch.nn.functional.silu(gate) * up

        return self.down_proj(hidden)

In [3]:
class LlamaAttention(torch.nn.Module):
    def __init__(self, num_kv_head, num_attn_head, head_dim, hidden_size, max_seq_len):
        super().__init__()

        self.num_kv_head = num_kv_head
        self.num_attn_head = num_attn_head
        self.head_dim = head_dim
        self.hidden_size = hidden_size
        self.max_seq_len = max_seq_len

        self.theta = 500000
        self.low_freq_factor = 1.0
        self.high_freq_factor = 4.0
        self.factor = 32.0
        self.original_max_position_embeddings = 8192

        self.k_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=num_kv_head * head_dim, bias=False
        )
        self.o_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=hidden_size, bias=False
        )
        self.q_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=hidden_size, bias=False
        )
        self.v_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=num_kv_head * head_dim, bias=False
        )

        self.register_buffer(
            "mask", torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        )

    def forward(self, x):

        B, S, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, S, self.num_attn_head, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.num_kv_head, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.num_kv_head, self.head_dim).transpose(1, 2)

        # Llama 3.1 RoPE
        # Base inverse frequencies
        inv_freq = 1.0 / (
            self.theta
            ** (
                torch.arange(
                    0,
                    self.head_dim,
                    2,
                    device=x.device,
                    dtype=torch.float32,
                )
                / self.head_dim
            )
        )

        # Wavelength of each frequency
        wavelen = 2 * torch.pi / inv_freq

        low_freq_wavelen = self.original_max_position_embeddings / self.low_freq_factor

        high_freq_wavelen = (
            self.original_max_position_embeddings / self.high_freq_factor
        )

        # Long-wavelength frequencies:
        # divide their inverse frequency by the scaling factor.
        inv_freq_llama = torch.where(
            wavelen > low_freq_wavelen,
            inv_freq / self.factor,
            inv_freq,
        )

        # Frequencies in the transition region
        smooth_factor = (
            self.original_max_position_embeddings / wavelen - self.low_freq_factor
        ) / (self.high_freq_factor - self.low_freq_factor)

        smoothed_inv_freq = (
            1 - smooth_factor
        ) * inv_freq_llama / self.factor + smooth_factor * inv_freq_llama

        # Transition region:
        # high_freq_wavelen <= wavelen <= low_freq_wavelen
        is_medium_freq = ~(wavelen < high_freq_wavelen) & ~(wavelen > low_freq_wavelen)

        inv_freq_llama = torch.where(
            is_medium_freq,
            smoothed_inv_freq,
            inv_freq_llama,
        )

        # Position × frequency
        positions = torch.arange(
            S,
            device=x.device,
            dtype=torch.float32,
        )

        freqs = torch.outer(
            positions,
            inv_freq_llama,
        )

        # cos/sin
        cos = freqs.cos()
        sin = freqs.sin()

        # [S, 64] → [S, 128]
        cos = torch.cat((cos, cos), dim=-1)
        sin = torch.cat((sin, sin), dim=-1)

        # Apply RoPE
        q = (q * cos) + (self.rotate_half(q) * sin)
        k = (k * cos) + (self.rotate_half(k) * sin)

        num_groups = self.num_attn_head // self.num_kv_head

        k = k.repeat_interleave(num_groups, dim=1)
        v = v.repeat_interleave(num_groups, dim=1)

        scores = torch.einsum("bhqd, bhkd -> bhqk", q, k) * (self.head_dim**-0.5)
        scores = scores.masked_fill(self.mask[:S, :S], float("-inf"))
        attn_weights = torch.softmax(scores, dim=-1)

        attn_output = torch.einsum("bhqk, bhkd -> bhqd", attn_weights, v)
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(B, S, self.hidden_size)

        return self.o_proj(attn_output)

    def rotate_half(self, x):
        x1 = x[..., : self.head_dim // 2]
        x2 = x[..., self.head_dim // 2 :]

        return torch.cat((-x2, x1), dim=-1)

In [4]:
class LlamaDecoderLayer(torch.nn.Module):
    def __init__(
        self,
        hidden_size,
        intermediate_size,
        num_kv_head,
        num_attn_head,
        head_dim,
        max_seq_len,
    ):
        super().__init__()
        self.input_layernorm = torch.nn.RMSNorm(hidden_size, eps=1e-5)
        self.self_attn = LlamaAttention(
            num_kv_head,
            num_attn_head,
            head_dim,
            hidden_size,
            max_seq_len,
        )
        self.post_attention_layernorm = torch.nn.RMSNorm(hidden_size, eps=1e-5)
        self.mlp = LlamaMlp(
            hidden_size,
            intermediate_size,
        )

    def forward(self, x):

        residual = x

        x = self.input_layernorm(x)
        x = self.self_attn(x)

        x = residual + x

        residual = x

        x = self.post_attention_layernorm(x)
        x = self.mlp(x)

        x = residual + x

        return x

In [48]:
class LlamaModel(torch.nn.Module):
    def __init__(
        self,
        files,
        num_layers,
        hidden_size,
        intermediate_size,
        num_kv_head,
        num_attn_head,
        head_dim,
        max_seq_len,
    ):
        super().__init__()

        self.num_layers = num_layers

        self.handles = [safe_open(path, framework="pt", device="cpu") for path in files]

        self.embed_tokens = torch.nn.Embedding(
            num_embeddings=128256, embedding_dim=3072
        )
        self.embed_tokens.weight.data.copy_(
            self.get_tensor(self.handles, "model.embed_tokens.weight")
        )

        self.decoder = LlamaDecoderLayer(
            hidden_size,
            intermediate_size,
            num_kv_head,
            num_attn_head,
            head_dim,
            max_seq_len,
        ).eval()

        self.norm = torch.nn.RMSNorm(hidden_size, eps=1e-5)
        self.norm.weight.data.copy_(self.get_tensor(self.handles, "model.norm.weight"))

    @torch.no_grad()
    def forward(self, input_ids):

        x = self.embed_tokens(input_ids)

        for layer_id in range(self.num_layers):
            self.load_layer(self.decoder, layer_id, self.handles)
            x = self.decoder(x)

        x = self.norm(x)

        logits = x @ self.embed_tokens.weight.T
        next_token_logits = logits[:, -1, :]

        return torch.argmax(next_token_logits, dim=-1)

    def get_tensor(self, handles, name):
        for handle in handles:
            if name in handle.keys():
                return handle.get_tensor(name)

        raise KeyError(f"Tensor not found: {name}")

    def load_layer(self, decoder, layer_id, handles):

        prefix = f"model.layers.{layer_id}."

        decoder.input_layernorm.weight.data.copy_(
            self.get_tensor(handles, prefix + "input_layernorm.weight")
        )

        decoder.self_attn.q_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "self_attn.q_proj.weight")
        )

        decoder.self_attn.k_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "self_attn.k_proj.weight")
        )

        decoder.self_attn.v_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "self_attn.v_proj.weight")
        )

        decoder.self_attn.o_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "self_attn.o_proj.weight")
        )

        decoder.post_attention_layernorm.weight.data.copy_(
            self.get_tensor(handles, prefix + "post_attention_layernorm.weight")
        )

        decoder.mlp.gate_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "mlp.gate_proj.weight")
        )

        decoder.mlp.up_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "mlp.up_proj.weight")
        )

        decoder.mlp.down_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "mlp.down_proj.weight")
        )

In [66]:
# let's load LlamaDecoderLayer for 32 times

hidden_size = 3072
head_dim = 128
num_attn_head = 24
num_kv_head = 8
max_seq_len = 100
theta = 500000
intermediate_size = 8192
layers = 28

model = LlamaModel(
    files,
    layers,
    hidden_size,
    intermediate_size,
    num_kv_head,
    num_attn_head,
    head_dim,
    max_seq_len,
)

In [75]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("./my_models/tokenizer.json")


def generate(model, tokenizer, prompt, max_new_tokens):

    encoded = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        [encoded.ids],
        dtype=torch.long,
    )

    for _ in range(max_new_tokens):

        next_token = model(input_ids)

        input_ids = torch.cat(
            [
                input_ids,
                next_token.unsqueeze(0),
            ],
            dim=1,
        )

        # Stop if EOS
        if next_token.item() == tokenizer.token_to_id("<|eot_id|>"):
            break

    return tokenizer.decode(input_ids[0].tolist())

In [77]:
text = generate(
    model,
    tokenizer,
    "write 'I love You' as it is!",
    max_new_tokens=50,
)

print(text)

write 'I love You' as it is! I love you!
write 'I love You' as it is!
I love you! I love you! I love you!1. I love you! I love you! I love you! I love you! I love you!
